# B. Cross-encoder bge-reranker-v2-m3 без дообучения: оценка всех пар (Kaggle, две GPU, float16)
Модель — BAAI/bge-reranker-v2-m3 как есть, без дообучения: дообучение на парах P1 ухудшило Recall@50 на 1000 запросах P2
(0.884 без дообучения, 0.850 после; см. README). Запускается дважды: на парах мест 1–150 и на парах мест 151–300 первого этапа.
Пары сортируются по длине текста, делятся на два шарда через один и оцениваются двумя процессами на двух GPU. Сохранение каждые 200 тыс. пар; скорость после первых 2000 пар, ниже 100 пар/с — громкое предупреждение.
Результат: `scores_<часть>.parquet` (query_id, item_id, ce_score).

In [ ]:
# Те же версии, что локально (requirements-lock.txt)
!pip install -q sentence-transformers==6.1.0 transformers==5.17.0 datasets==5.0.1 accelerate==1.15.0

In [ ]:
COMMON_SRC = '\nimport time\nimport numpy as np\nimport torch\n# Только bge-reranker-v2-m3: Qwen3-Reranker убран (OOM на Kaggle при 384 токенах, bge сильнее на сравнении).\nMODEL = "intfloat/multilingual-e5-small" if LOCAL_SMOKE else "BAAI/bge-reranker-v2-m3"\n# Длина по замеру на 20 тыс. пар P2 (токенизатор XLM-R): длиннее 192 — 24.9% пар, длиннее 256 — 0.7%.\nMAX_LEN = 256\n# Инференс в float16 на GPU (в float32 было 22 пары/с); на CPU при локальной проверке — float32.\nDTYPE = torch.float16 if torch.cuda.is_available() else torch.float32\nMIN_RATE = 100   # пар/с: ниже — громкое предупреждение\nFIRST_N = 50 if LOCAL_SMOKE else 2000\n\ndef load_ce(path, dtype=None):\n    from sentence_transformers.cross_encoder import CrossEncoder\n    kw = {"model_kwargs": {"dtype": dtype}} if dtype is not None else {}\n    return CrossEncoder(path, max_length=MAX_LEN, num_labels=1 if LOCAL_SMOKE else None, **kw)\n\ndef predict_sorted(ce, queries, docs, batch, tag=""):\n    """Оценка пар, отсортированных по длине (меньше паддинга); возвращает в исходном порядке.\n    После первых FIRST_N пар печатает скорость, ниже MIN_RATE — громкое предупреждение."""\n    q, d = list(queries), list(docs)\n    order = np.argsort([len(a) + len(b) for a, b in zip(q, d)], kind="stable")\n    out = np.empty(len(q), dtype=np.float32)\n    t = time.time()\n    head, tail = order[:FIRST_N], order[FIRST_N:]\n    out[head] = ce.predict([[q[i], d[i]] for i in head], batch_size=batch, show_progress_bar=False)\n    rate = len(head) / (time.time() - t)\n    msg = f"{tag} СКОРОСТЬ после {len(head)} пар: {rate:.0f} пар/с"\n    if rate < MIN_RATE:\n        msg = "!" * 60 + f"\\n!!! ВНИМАНИЕ: {msg} < {MIN_RATE}. Проверьте GPU и float16 !!!\\n" + "!" * 60\n    print(msg, flush=True)\n    if len(tail):\n        out[tail] = ce.predict([[q[i], d[i]] for i in tail], batch_size=batch, show_progress_bar=False)\n    return out\n'
WORKER_SRC = '\nimport sys, os\nshard, gpu, model_path, parts, in_dir, out_dir, budget_h, smoke = sys.argv[1:9]\nos.environ["CUDA_VISIBLE_DEVICES"] = gpu\nos.environ["TOKENIZERS_PARALLELISM"] = "false"\nLOCAL_SMOKE = smoke == "1"\nimport pandas as pd\nexec(open(os.path.join(out_dir, "common.py")).read())\nT0 = time.time()\nCHUNK = 200 if LOCAL_SMOKE else 200_000   # сохранение каждые CHUNK пар\nBATCH = 16 if LOCAL_SMOKE else 256\nce = load_ce(model_path, DTYPE)           # float16 на GPU\nfor part in parts.split(","):\n    df = pd.read_parquet(os.path.join(in_dir, f"shard{shard}_{part}.parquet"))   # уже отсортирован по длине\n    done, t_part = [], time.time()\n    for s0 in range(0, len(df), CHUNK):\n        if (time.time() - T0) / 3600 > float(budget_h):\n            print(f"[{shard}] бюджет исчерпан на {part}: {s0}/{len(df)}", flush=True); break\n        c = df.iloc[s0:s0 + CHUNK]\n        sc = predict_sorted(ce, c["query_text"], c["item_text"], BATCH, f"[{shard}] {part}") if s0 == 0 else \\\n            ce.predict([[a, b] for a, b in zip(c["query_text"], c["item_text"])], batch_size=BATCH, show_progress_bar=False)\n        done.append(c[["order"]].assign(ce_score=np.asarray(sc, dtype=np.float32)))\n        pd.concat(done).to_parquet(os.path.join(out_dir, f"scores_{part}_shard{shard}.parquet"), index=False)\n        n = min(s0 + CHUNK, len(df))\n        print(f"[{shard}] {part}: {n}/{len(df)}, {n / (time.time() - t_part):.0f} пар/с, прошло {(time.time() - T0) / 3600:.2f} ч", flush=True)\nprint(f"[{shard}] готово", flush=True)\n'
import os, glob, json, subprocess, sys
LOCAL_SMOKE = os.environ.get("LOCAL_SMOKE") == "1"
BUDGET_H = 9.0
PARTS = ["P2", "P3", "bench"]
def find(name, root=None):
    root = root or (os.environ.get("RERANK_DIR") if LOCAL_SMOKE else "/kaggle/input")
    hits = glob.glob(f"{root}/**/{name}", recursive=True)
    return hits[0] if hits else None
WORK = os.environ.get("WORK_B", "/tmp/rerank_B") if LOCAL_SMOKE else "/kaggle/working"
os.makedirs(WORK, exist_ok=True)
import time
import numpy as np
import torch
# Только bge-reranker-v2-m3: Qwen3-Reranker убран (OOM на Kaggle при 384 токенах, bge сильнее на сравнении).
MODEL = "intfloat/multilingual-e5-small" if LOCAL_SMOKE else "BAAI/bge-reranker-v2-m3"
# Длина по замеру на 20 тыс. пар P2 (токенизатор XLM-R): длиннее 192 — 24.9% пар, длиннее 256 — 0.7%.
MAX_LEN = 256
# Инференс в float16 на GPU (в float32 было 22 пары/с); на CPU при локальной проверке — float32.
DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32
MIN_RATE = 100   # пар/с: ниже — громкое предупреждение
FIRST_N = 50 if LOCAL_SMOKE else 2000

def load_ce(path, dtype=None):
    from sentence_transformers.cross_encoder import CrossEncoder
    kw = {"model_kwargs": {"dtype": dtype}} if dtype is not None else {}
    return CrossEncoder(path, max_length=MAX_LEN, num_labels=1 if LOCAL_SMOKE else None, **kw)

def predict_sorted(ce, queries, docs, batch, tag=""):
    """Оценка пар, отсортированных по длине (меньше паддинга); возвращает в исходном порядке.
    После первых FIRST_N пар печатает скорость, ниже MIN_RATE — громкое предупреждение."""
    q, d = list(queries), list(docs)
    order = np.argsort([len(a) + len(b) for a, b in zip(q, d)], kind="stable")
    out = np.empty(len(q), dtype=np.float32)
    t = time.time()
    head, tail = order[:FIRST_N], order[FIRST_N:]
    out[head] = ce.predict([[q[i], d[i]] for i in head], batch_size=batch, show_progress_bar=False)
    rate = len(head) / (time.time() - t)
    msg = f"{tag} СКОРОСТЬ после {len(head)} пар: {rate:.0f} пар/с"
    if rate < MIN_RATE:
        msg = "!" * 60 + f"\n!!! ВНИМАНИЕ: {msg} < {MIN_RATE}. Проверьте GPU и float16 !!!\n" + "!" * 60
    print(msg, flush=True)
    if len(tail):
        out[tail] = ce.predict([[q[i], d[i]] for i in tail], batch_size=batch, show_progress_bar=False)
    return out

open(f"{WORK}/common.py", "w").write("LOCAL_SMOKE = " + repr(LOCAL_SMOKE) + "\n" + COMMON_SRC)
open(f"{WORK}/worker.py", "w").write(WORKER_SRC)

In [ ]:
import pandas as pd
MODEL_PATH = MODEL   # без дообучения
print("модель", MODEL_PATH, "| max_len", MAX_LEN, "| dtype", DTYPE)

# Шарды: сортировка по длине, затем через один — у обоих процессов одинаковая смесь длин
for part in PARTS:
    df = pd.read_parquet(find(f"pairs_{part}.parquet"), columns=["query_id", "item_id", "query_text", "item_text"])
    if LOCAL_SMOKE:
        df = df.head(300)
    df["order"] = np.arange(len(df))
    df = df.iloc[np.argsort((df["query_text"].str.len() + df["item_text"].str.len()).to_numpy(), kind="stable")]
    for k in (0, 1):
        df.iloc[k::2].to_parquet(f"{WORK}/shard{k}_{part}.parquet", index=False)
    df[["query_id", "item_id", "order"]].to_parquet(f"{WORK}/ids_{part}.parquet", index=False)

In [ ]:
# Два процесса, по одному на GPU; прогресс печатается по ходу
procs = []
for k in (0, 1):
    log = open(f"{WORK}/worker{k}.log", "w")
    procs.append(subprocess.Popen([sys.executable, f"{WORK}/worker.py", str(k), "" if LOCAL_SMOKE else str(k), MODEL_PATH,
                                   ",".join(PARTS), WORK, WORK, str(BUDGET_H - 0.2), "1" if LOCAL_SMOKE else "0"],
                                  stdout=log, stderr=subprocess.STDOUT))
seen = [0, 0]
while any(p.poll() is None for p in procs):
    time.sleep(5 if LOCAL_SMOKE else 60)
    for k in (0, 1):
        lines = [l for l in open(f"{WORK}/worker{k}.log").read().splitlines() if l.startswith(f"[{k}]") or "ВНИМАНИЕ" in l]
        for l in lines[seen[k]:]:
            print(l, flush=True)
        seen[k] = len(lines)
print("коды выхода:", [p.returncode for p in procs])
for k in (0, 1):
    print(open(f"{WORK}/worker{k}.log").read()[-1500:])

In [ ]:
# Сборка: оценки шардов по исходному порядку пар
for part in PARTS:
    ids = pd.read_parquet(f"{WORK}/ids_{part}.parquet")
    sc = pd.concat([pd.read_parquet(p) for p in sorted(glob.glob(f"{WORK}/scores_{part}_shard*.parquet"))])
    out = ids.merge(sc, on="order", how="left").sort_values("order")[["query_id", "item_id", "ce_score"]]
    print(f"{part}: оценено {out['ce_score'].notna().mean():.4f} пар из {len(out)}")
    out.to_parquet(f"{WORK}/scores_{part}.parquet", index=False)
json.dump({"mode": "zeroshot", "model": MODEL_PATH, "max_len": MAX_LEN, "dtype": str(DTYPE)}, open(f"{WORK}/score_meta.json", "w"))
print(sorted(os.listdir(WORK)))